# 4. Multivariable Cox Regression

Multivariable Cox proportional hazards model with Schoenfeld residual testing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Load and prepare data
df = pd.read_csv('LiverMets_Final_Dataset.csv')

complete_tnm = df[
    (df['T_STAGE'].notna()) & (df['T_STAGE'] != 'ND') &
    (df['N_STAGE'].notna()) & (df['N_STAGE'] != 'ND') &
    (df['M_STAGE'].notna()) & (df['M_STAGE'] != 'ND')
]
included = complete_tnm[
    (complete_tnm['SURVIVAL_YEARS'].notna()) & 
    (complete_tnm['SURVIVAL_YEARS'] > 0) &
    (complete_tnm['VITAL_STATUS'].notna())
]

# Define phenotypes
df_cox = included.copy()
df_cox['PHENOTYPE'] = np.nan
mask1 = (df_cox['M_STAGE'] == 'M0') & (df_cox['N_STAGE'].isin(['N0', 'N1']))
df_cox.loc[mask1, 'PHENOTYPE'] = 1
mask2a = (df_cox['M_STAGE'] == 'M0') & (df_cox['N_STAGE'] == 'N2')
mask2b = (df_cox['M_STAGE'] == 'M1') & (df_cox['N_STAGE'].isin(['N0', 'N1']))
df_cox.loc[mask2a | mask2b, 'PHENOTYPE'] = 2
mask3 = (df_cox['M_STAGE'] == 'M1') & (df_cox['N_STAGE'] == 'N2')
df_cox.loc[mask3, 'PHENOTYPE'] = 3

print(f"Total cohort: {len(df_cox):,}")

## Complete-Case Analysis for Cox Regression

In [ ]:
# Select variables for Cox model
cox_vars = ['AGE_AT_REFERRAL', 'GENDER', 'PHENOTYPE', 'NB_METS_GROUP', 'TREATMENT']

# Complete case analysis
df_cox_complete = df_cox.dropna(subset=cox_vars + ['SURVIVAL_YEARS', 'VITAL_STATUS'])

print(f"Original cohort: {len(df_cox):,}")
print(f"Complete case (n_cox): {len(df_cox_complete):,}")
print(f"Missing data: {len(df_cox) - len(df_cox_complete):,}")

# Prepare variables
df_cox_complete['AGE_centered'] = df_cox_complete['AGE_AT_REFERRAL'] - df_cox_complete['AGE_AT_REFERRAL'].mean()
df_cox_complete['MALE'] = (df_cox_complete['GENDER'].str.upper() == 'MALE').astype(int)

# Phenotype dummies (reference: Phenotype 1)
df_cox_complete['PH2'] = (df_cox_complete['PHENOTYPE'] == 2).astype(int)
df_cox_complete['PH3'] = (df_cox_complete['PHENOTYPE'] == 3).astype(int)

print("\nVariables prepared for Cox model")

## Fit Multivariable Cox Model

In [ ]:
from lifelines import CoxPHFitter

# Prepare data for Cox model
cox_data = df_cox_complete[['SURVIVAL_YEARS', 'VITAL_STATUS', 'AGE_centered', 'MALE', 'PH2', 'PH3']].copy()

# Fit Cox model
cph = CoxPHFitter()
cph.fit(cox_data, duration_col='SURVIVAL_YEARS', event_col='VITAL_STATUS')

# Display summary
print("Multivariable Cox Proportional Hazards Model")
print("="*70)
print(cph.summary)
print(f"\nConcordance index: {cph.concordance_index_:.3f}")

## Schoenfeld Residual Test (Proportional Hazards Assumption)

In [ ]:
from lifelines.statistics import proportional_hazard_test

# Test proportional hazards assumption
ph_results = proportional_hazard_test(cph, cox_data, time_transform='rank')

print("\nSchoenfeld Residual Test: Proportional Hazards Assumption")
print("="*70)
print(ph_results)

print("\nInterpretation:")
print("  - p-value > 0.05: Assumption satisfied for that variable")
print("  - p-value < 0.05: Evidence of non-proportional hazards")
print("    (but variable may be retained if clinically relevant)")

## Hazard Ratios and Confidence Intervals

In [ ]:
# Extract HRs and CIs
results_hr = cph.summary.copy()
results_hr['HR'] = np.exp(results_hr['coef'])
results_hr['CI_lower'] = np.exp(results_hr['coef lower 95%'])
results_hr['CI_upper'] = np.exp(results_hr['coef upper 95%'])

print("\nHazard Ratios (HR) and 95% Confidence Intervals")
print("="*70)
for idx in results_hr.index:
    hr = results_hr.loc[idx, 'HR']
    ci_l = results_hr.loc[idx, 'CI_lower']
    ci_u = results_hr.loc[idx, 'CI_upper']
    p_val = results_hr.loc[idx, 'p']
    print(f"{idx:20} HR: {hr:6.3f}  95% CI: ({ci_l:.3f}–{ci_u:.3f})  p={p_val:.4f}")

## Forest Plot

In [ ]:
# Forest plot
fig, ax = plt.subplots(figsize=(10, 6))

results_plot = results_hr.iloc[:-1]  # Exclude intercept if present
y_pos = np.arange(len(results_plot))

ax.scatter(results_plot['HR'], y_pos, s=100, color='darkblue', zorder=3)
ax.hlines(y_pos, results_plot['CI_lower'], results_plot['CI_upper'], colors='darkblue', linewidth=2)
ax.axvline(1.0, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='No effect (HR=1)')

ax.set_yticks(y_pos)
ax.set_yticklabels(results_plot.index)
ax.set_xlabel('Hazard Ratio (log scale)', fontsize=12, fontweight='bold')
ax.set_xscale('log')
ax.grid(True, alpha=0.3, axis='x')
ax.legend()
ax.set_title('Multivariable Cox Model: Forest Plot', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()